In [13]:
import os
import json
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
groq_api_key=os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("GROQ API KEY exists")
else:
    print("GROQ API Key does not exists")

GROQ API KEY exists


In [15]:
groq_base_url=os.getenv("GROQ_BASE_URL")


In [16]:
system_message="""You are a helpful assissant for an Airline called FlightAI. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you don't know the answer, say so.
"""

In [17]:
from openai import OpenAI

In [22]:
def chat(message, history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]

    messages=[{"role":"system", "content":system_message}]+history+[{"role":"user", "content":"message"}]

    openai=OpenAI(api_key=groq_api_key, base_url=groq_base_url)

    response=openai.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
    )

    return response.choices[0].message.content



In [23]:
import gradio as gr 
gr.ChatInterface(fn=chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://b6f26729d0d1fa967c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [59]:
ticket_prices={"london":"$799", "paris":"$899", "tokoyo":"$1499", "berlin":"$499"}



In [60]:
def get_ticket_prices(destination_city):
    print(f'Tool called for city {destination_city}')
    price=ticket_prices.get(destination_city,"Unknown city")
    return f'The price of a ticket is {destination_city} is {price}'

In [61]:
get_ticket_prices("london")

Tool called for city london


'The price of a ticket is london is $799'

In [94]:
price_function={
    "name":"get_ticket_prices",
    "description":"Get the price of a return ticket to the destination city",
    "parameters":{
        "type":"object",
        "proprties":{
            "destination_city":{
                "type":"string",
                "description":"The city that the customer wants to travel to"
            },
        },
        "required":["destination_city"],
        "additionalProperties":False
    }
}

In [95]:
tools=[{"type":"function", "function":price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_prices',
   'description': 'Get the price of a return ticket to the destination city',
   'parameters': {'type': 'object',
    'proprties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [96]:
def chat(message, history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]

    messages=[{"role":"system", "content":system_message}]+history+[{"role":"user", "content":message}]

    openai=OpenAI(api_key=groq_api_key, base_url=groq_base_url)
    response=openai.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
        )
    print(response)
    
    if response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response=openai.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
        )
    return response.choices[0].message.content



In [102]:
def handle_tool_call(message):
    tool_call=message.tool_calls[0]
    if tool_call.function.name=="get_ticket_prices":
        arguments=json.loads(tool_call.function.arguments)
        city=arguments.get('destination_city')
        price_details=get_ticket_prices(city)
        response={
            "role":"tool",
            "content":price_details,
            "tool_call_id":tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7883
* Running on public URL: https://5b79aef0b10cf0393d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/workspaces/AI-Engineer-Learn/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/AI-Engineer-Learn/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/AI-Engineer-Learn/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/AI-Engineer-Learn/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/AI-Engineer-Learn/.venv/lib/python3.12/site-packages/gradio/utils.py", line 1027, in async_w